In [ ]:
import pandas as pd
import sqlalchemy
import os
import dotenv
dotenv.load_dotenv()
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost
from matplotlib import pyplot as plt
import pickle

# Conexion y Extracción de Datos

In [5]:
server = os.environ.get("SQL_SERVER", "")
db = os.environ.get("SQL_DB", "")
pwd = os.environ.get("SQL_PWD", "")
user = os.environ.get("SQL_USER","")

conn_str = f"mssql+pymssql://{user}:{(pwd)}@{server}/{db}"

engine = sqlalchemy.create_engine(conn_str)
customer_df = pd.read_sql("SELECT CustomerID, Title, Suffix, SalesPerson, ModifiedDate FROM SalesLT.Customer", engine)



# Dividir en Train y Test

In [6]:
# Convertir fecha en segundos para regresion Unix
unix_epoch = pd.Timestamp("1970-01-01")
customer_df["ModifiedDate"] = (customer_df["ModifiedDate"] - unix_epoch).dt.total_seconds().astype('int64')
customer_df = pd.get_dummies(customer_df, columns=["Title", "Suffix", "SalesPerson"], drop_first=True)
x_train, x_test, y_train, y_test = train_test_split(customer_df.drop("ModifiedDate", axis=1), customer_df["ModifiedDate"].astype('int64'), test_size=0.2, random_state=42)

In [7]:
model = xgboost.XGBRegressor( n_estimators=20)
model.fit(x_train, y_train)
model_pred = model.predict(x_test)
model_mse = mean_squared_error(y_test, model_pred)
model_r2 = r2_score(y_test, model_pred)
print(f"XGBoost MSE: {model_mse}, R2: {model_r2}")
# Es un modelo malo porque las variables no tienen nada que ver con cuando se modifica un registro
pickle.dump(model, open("xgboost_model.pkl", "wb"))

XGBoost MSE: 759059127061113.4, R2: -0.03288102149963379
